In [1]:
!pip install -U chromadb langchain langchain-groq langchain-community \
    langchain-chroma langchain-text-splitters transformers \
    sentence-transformers unstructured "unstructured[pdf]"

In [2]:
#!apt-get install poppler-utils

In [3]:
import os

from langchain.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

In [4]:
#!pip install --upgrade unstructured[local-inference]

In [5]:
os.environ["GROQ_API_KEY"] = "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"

In [6]:
#fetch the pdf from url
import requests
url = "https://courses.cs.umbc.edu/671/fall12/notes/python/04_python_functions.pdf"

response = requests.get(url)

In [7]:
response

<Response [200]>

In [8]:
#save the pdf local file
with open("python_inbuildfunctions.pdf", "wb") as f:
   f.write(response.content)

In [9]:
from langchain_community.document_loaders import UnstructuredFileIOLoader

with open("python_inbuildfunctions.pdf", "rb") as f:
    loader = UnstructuredFileIOLoader(file=f)
    documents = loader.load()

documents  # This will be a list of Document objects

/tmp/ipython-input-9-1778148066.py:4: LangChainDeprecationWarning: The class `UnstructuredFileIOLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-unstructured package and should be used instead. To use it run `pip install -U :class:`~langchain-unstructured` and import as `from :class:`~langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileIOLoader(file=f)


[Document(metadata={}, page_content='Functions in Python\n\nDefining Functions\n\nFunction definition begins with “def.” Function name and its arguments.\n\ndef get_final_answer(filename): “““Documentation String”””\n\nline1 line2 return total_counter\n\nThe indentation matters… First line with less indentation is considered to be outside of the function definition.\n\nThe keyword ‘return’ indicates the value to be sent back to the caller.\n\nNo header file or declaration of types of function or arguments\n\nColon.\n\nPython and Types\n\nDynamic typing: Python determines the data types of variable bindings in a program automatically\n\nStrong typing: But Python’s not casual about types, it enforces the types of objects\n\nFor example, you can’t just append an integer to a string, but must first convert it to a string\n\nx = “the answer is ” # x bound to a string y = 23 # y bound to an integer. print x + y # Python will complain!\n\nCalling a Function\n\nThe syntax for a function call i

In [10]:
text_splitter = CharacterTextSplitter(chunk_size = 100, chunk_overlap = 50)

In [11]:
text_splitter

In [12]:
texts = text_splitter.split_documents(documents)

In [13]:
type(texts)

list

In [14]:
texts[10]

Document(metadata={}, page_content='Calling a Function')

In [15]:
embeddings = HuggingFaceEmbeddings()

/tmp/ipython-input-15-3655315981.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipython-input-15-3655315981.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https:

In [16]:
persist_directory = "vector_db"

In [17]:
vectordb = Chroma.from_documents(texts, embeddings, persist_directory=persist_directory)

In [18]:
# retrival

retriever = vectordb.as_retriever()

In [19]:
llm = ChatGroq(model = "llama-3.3-70b-versatile",temperature=0)

In [20]:
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever,return_source_documents=True)

In [28]:
query = 'explain about filter, map, reduce'
response = qa_chain.invoke({"query":query})

In [29]:
print(response)

{'query': 'explain about filter, map, reduce', 'result': "`filter`, `map`, and `reduce` are three fundamental concepts in functional programming that can be used to process and transform data in a declarative way.\n\n1. **Filter**: The `filter` function takes a predicate (a function that returns a boolean value) and a collection of items as input. It returns a new collection that includes only the items for which the predicate returns `True`. In other words, it filters out the items that do not meet the condition.\n\n2. **Map**: The `map` function takes a transformation function and a collection of items as input. It applies the transformation function to each item in the collection and returns a new collection with the results. This allows you to transform or convert each item in the collection in a consistent way.\n\n3. **Reduce**: The `reduce` function takes a reduction function and a collection of items as input. It applies the reduction function to the first two items in the colle

In [30]:
print(response['result'])

`filter`, `map`, and `reduce` are three fundamental concepts in functional programming that can be used to process and transform data in a declarative way.

1. **Filter**: The `filter` function takes a predicate (a function that returns a boolean value) and a collection of items as input. It returns a new collection that includes only the items for which the predicate returns `True`. In other words, it filters out the items that do not meet the condition.

2. **Map**: The `map` function takes a transformation function and a collection of items as input. It applies the transformation function to each item in the collection and returns a new collection with the results. This allows you to transform or convert each item in the collection in a consistent way.

3. **Reduce**: The `reduce` function takes a reduction function and a collection of items as input. It applies the reduction function to the first two items in the collection, then to the result and the next item, and so on, until it